# NLP, TP 3 : classer les répliques par prompting

Le problème reste celui des TP précédents : retrouver lequel des 5 personnages (DUKE VINCENTIO, GLOUCESTER, MENENIUS, PETRUCHIO, ROMEO) prononce une réplique de *Tiny Shakespeare*.

Changement de méthode : on n'entraîne plus rien. On décrit la tâche à un LLM dans un prompt et on lit sa réponse. Le notebook suit les quatre étapes de la consigne :

| Étape | Où |
|---|---|
| Prompting | §3 : trois prompts (zero-shot, zero-shot avec contexte, few-shot) |
| Modèle | §2 : un modèle ouvert chargé dans Colab, ou une API |
| Sortie générée | §4 : réponse en JSON, parsing, comptage des sorties invalides |
| Évaluation | §5 à §8 : mêmes métriques et même test que les TP 1 et 2, comparaison avec la baseline |

Protocole : les trois prompts sont comparés sur la validation, on garde le meilleur, et on ne le lance qu'une fois sur le test.

## 0. Environnement

Sous Colab, choisir un GPU (*Exécution → Modifier le type d'exécution → T4*) avant de lancer.

In [ ]:
%pip install -q -U transformers accelerate

In [ ]:
import re
import json
import time
import random
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 100)

## 1. Les données, à l'identique des TP précédents

Pour que les scores soient comparables, on reconstruit exactement le même test : mêmes filtres, même random_state, même stratification. Le train ne sert plus à entraîner le LLM ; il fournit les exemples du prompt few-shot et sert à réentraîner la baseline TF-IDF.

In [ ]:
URL_CORPUS = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
corpus = urllib.request.urlopen(URL_CORPUS).read().decode("utf-8")


def extraire_repliques(texte: str) -> pd.DataFrame:
    """Coupe le texte en blocs séparés par une ligne vide et garde ceux qui commencent par 'NOM:'."""
    lignes = []
    for bloc in texte.split("\n\n"):
        bloc = bloc.strip()
        if not bloc:
            continue
        premiere, *suite = bloc.split("\n")
        entete = re.match(r"^([A-Z][A-Za-z' ]*):$", premiere)
        if entete and suite:
            lignes.append({
                "locuteur": entete.group(1).strip(),
                "replique": " ".join(l.strip() for l in suite).strip(),
            })
    return pd.DataFrame(lignes)



N_LOCUTEURS, MIN_MOTS = 5, 5
repliques = extraire_repliques(corpus)
repliques["n_mots"] = repliques["replique"].str.split().str.len()
top_locuteurs = repliques["locuteur"].value_counts().head(N_LOCUTEURS).index
garde = repliques["locuteur"].isin(top_locuteurs) & (repliques["n_mots"] >= MIN_MOTS)
donnees = repliques.loc[garde, ["replique", "locuteur"]].reset_index(drop=True)

# split stratifié 70 / 15 / 15, identique au TP 2
X_train, X_temp, y_train, y_temp = train_test_split(
    donnees["replique"], donnees["locuteur"], test_size=0.30,
    stratify=donnees["locuteur"], random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE)

CANDIDATS = sorted(y_train.unique())
print(f"{len(donnees)} répliques : train {len(X_train)}, val {len(X_val)}, test {len(X_test)}")
print("classes :", CANDIDATS)
print(f"accuracy en répondant toujours la classe majoritaire (test) : {y_test.value_counts(normalize=True).max():.1%}")

## 2. Le modèle

La variable BACKEND choisit entre deux options.

"hf" (par défaut) charge Qwen2.5-3B-Instruct, un modèle à poids ouverts, directement dans Colab. C'est gratuit et sans clé, mais il faut un GPU. Qwen/Qwen2.5-1.5B-Instruct va plus vite, au prix d'une qualité moindre.

"anthropic" appelle une API hébergée. Il faut une clé ANTHROPIC_API_KEY dans les *Secrets* de Colab. L'infrastructure est plus simple, mais c'est payant et on dépend du fournisseur.

Dans les deux cas, la suite du notebook n'utilise qu'une fonction, generer(conversations, temperature, max_new_tokens). On génère à température 0 (décodage glouton) : pour classer, on veut une réponse stable et reproductible.

In [ ]:
BACKEND = "hf"                          # "hf" ou "anthropic"
MODEL_HF = "Qwen/Qwen2.5-3B-Instruct"   # utilisé si BACKEND == "hf"
MODEL_API = "claude-haiku-5-5"          # utilisé si BACKEND == "anthropic"

In [ ]:
if BACKEND == "hf":
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    print("GPU disponible :", torch.cuda.is_available())
    tokenizer = AutoTokenizer.from_pretrained(MODEL_HF, padding_side="left")   # padding à gauche pour générer en batch
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    llm = AutoModelForCausalLM.from_pretrained(
        MODEL_HF,
        dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )
    llm.eval()

    def generer(conversations, temperature=0.0, max_new_tokens=30, batch_size=8):
        """Une conversation = liste de messages {role, content}. Renvoie le texte généré pour chacune."""
        reponses = []
        for debut in range(0, len(conversations), batch_size):
            lot = conversations[debut:debut + batch_size]
            prompts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=True) for c in lot]
            entree = tokenizer(prompts, return_tensors="pt", padding=True).to(llm.device)
            options = dict(max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id)
            if temperature > 0:
                options.update(do_sample=True, temperature=temperature, top_p=0.95)
            else:
                options.update(do_sample=False)
            with torch.no_grad():
                sortie = llm.generate(**entree, **options)
            nouveaux = sortie[:, entree["input_ids"].shape[1]:]       # on retire le prompt
            reponses += tokenizer.batch_decode(nouveaux, skip_special_tokens=True)
        return [r.strip() for r in reponses]

    def nb_tokens(conversation):
        texte = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
        return len(tokenizer(texte)["input_ids"])

elif BACKEND == "anthropic":
    import os, subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)
    from anthropic import Anthropic
    try:
        from google.colab import userdata
        cle = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        cle = os.environ.get("ANTHROPIC_API_KEY")
    client = Anthropic(api_key=cle)

    def generer(conversations, temperature=0.0, max_new_tokens=30, batch_size=None):
        reponses = []
        for conv in conversations:
            rep = client.messages.create(
                model=MODEL_API, max_tokens=max_new_tokens, temperature=temperature,
                system=conv[0]["content"], messages=[{"role": "user", "content": conv[1]["content"]}],
            )
            reponses.append(rep.content[0].text.strip())
        return reponses

    def nb_tokens(conversation):
        return int(sum(len(m["content"]) for m in conversation) / 4)   # estimation : ~4 caractères par token

else:
    raise ValueError("BACKEND doit valoir 'hf' ou 'anthropic'")

In [ ]:
# test rapide : le modèle répond-il ?
generer([[{"role": "system", "content": "Réponds en un mot."},
          {"role": "user", "content": "Quelle est la capitale de la France ?"}]])

## 3. Les prompts

On traite le prompt comme une spécification qu'on peut tester : objectif, contexte, contraintes, format de sortie, exemples. Trois versions, de la plus courte à la plus riche :

| Variante | Ce qu'elle contient | Ce qu'on mesure |
|---|---|---|
| zero_shot | la consigne et les 5 noms | ce que le LLM sait déjà de Shakespeare |
| zero_shot_contexte | en plus, la pièce et le rôle de chaque personnage | l'effet d'une connaissance donnée dans le prompt |
| few_shot | en plus, 2 vraies répliques par personnage, prises dans le train | l'effet des démonstrations |

Deux choix pour la robustesse. La réponse doit être un JSON {"locuteur": "&lt;NOM&gt;"}, facile à lire et à valider. Et chaque réplique est coupée à 120 mots : le contexte a un coût (tokens, temps de réponse), et le début d'une réplique suffit largement.

In [ ]:
MAX_MOTS_REPLIQUE = 120
K_SHOTS = 2   # exemples par personnage dans le prompt few-shot

DESCRIPTIONS = {
    "DUKE VINCENTIO": "le duc de Vienne dans « Measure for Measure » (Mesure pour mesure), qui se déguise en frère",
    "GLOUCESTER": "le duc de Gloucester (notamment le futur Richard III) dans « Richard III » et « Henry VI »",
    "MENENIUS": "Menenius Agrippa, patricien et ami de Coriolan dans « Coriolanus »",
    "PETRUCHIO": "Petruchio, mari de Catherine dans « The Taming of the Shrew » (La Mégère apprivoisée)",
    "ROMEO": "Roméo dans « Romeo and Juliet »",
}
NOMS = ", ".join(CANDIDATS)

SYSTEM = ("Tu es un expert de l'œuvre de Shakespeare. "
          "Tu réponds uniquement par un objet JSON valide, sans aucun texte autour.")


def tronquer(replique, n_mots=MAX_MOTS_REPLIQUE):
    mots = replique.split()
    return " ".join(mots[:n_mots]) + (" [...]" if len(mots) > n_mots else "")


# démonstrations few-shot : uniquement dans le train, répliques de 8 à 40 mots
train_df = pd.DataFrame({"replique": X_train.values, "locuteur": y_train.values})
train_df["n_mots"] = train_df["replique"].str.split().str.len()
demos = []
for nom in CANDIDATS:
    pool = train_df[(train_df["locuteur"] == nom) & train_df["n_mots"].between(8, 40)]
    demos += pool.sample(K_SHOTS, random_state=RANDOM_STATE)[["replique", "locuteur"]].values.tolist()
random.Random(RANDOM_STATE).shuffle(demos)   # on mélange pour ne pas regrouper les exemples par classe

CONSIGNE = ("Voici une réplique extraite d'une pièce de Shakespeare. "
            "Quel personnage la prononce ?\n"
            f"Choisis exactement un nom parmi : {NOMS}.\n"
            'Réponds uniquement avec un JSON de la forme {"locuteur": "<NOM>"}.')
BLOC_PERSONNAGES = "\n\nPersonnages possibles :\n" + "\n".join(
    f"- {nom} : {desc}" for nom, desc in DESCRIPTIONS.items())
BLOC_EXEMPLES = "\n\nExemples :\n\n" + "\n\n".join(
    f"Réplique : « {tronquer(r)} »\nRéponse : " + json.dumps({"locuteur": l}) for r, l in demos)


def construire_prompt(replique, variante):
    """Renvoie la conversation (system + user) envoyée au LLM pour une réplique et une variante."""
    supplement = ""
    if variante in ("zero_shot_contexte", "few_shot"):
        supplement += BLOC_PERSONNAGES
    if variante == "few_shot":
        supplement += BLOC_EXEMPLES
    user = f"{CONSIGNE}{supplement}\n\nRéplique : « {tronquer(replique)} »\nRéponse :"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]


VARIANTES = ["zero_shot", "zero_shot_contexte", "few_shot"]
print(construire_prompt(X_val.iloc[0], "few_shot")[1]["content"])

## 4. Lire et valider la sortie

Un LLM génère du texte de façon probabiliste : le JSON peut être mal formé, entouré de phrases, ou contenir un nom qui n'est pas dans la liste. On valide donc chaque réponse avant de l'utiliser, dans cet ordre :

1. chercher un objet JSON et lire son champ locuteur ;
2. sinon, chercher dans le texte un des 5 noms ou un alias courant (« Duke », « Richard »...) ;
3. sinon, étiqueter INCONNU. Ce cas compte comme une erreur, il n'est jamais retiré du calcul.

In [ ]:
ALIAS = {"DUKE": "DUKE VINCENTIO", "VINCENTIO": "DUKE VINCENTIO", "RICHARD": "GLOUCESTER"}


def vers_candidat(valeur):
    """Ramène une chaîne à l'un des 5 candidats, ou None si c'est impossible ou ambigu."""
    v = str(valeur).upper().strip()
    if v in CANDIDATS:
        return v
    trouves = {c for c in CANDIDATS if c in v} | {cible for alias, cible in ALIAS.items() if alias in v}
    return trouves.pop() if len(trouves) == 1 else None


def lire_reponse(sortie):
    """Renvoie (label, format_ok) ; format_ok = JSON valide contenant exactement un nom de la liste."""
    m = re.search(r"\{.*?\}", sortie, flags=re.S)
    if m:
        try:
            valeur = json.loads(m.group(0)).get("locuteur", "")
        except (json.JSONDecodeError, AttributeError):
            valeur = None
        if valeur is not None:
            nom = vers_candidat(valeur)
            if nom:
                return nom, str(valeur).upper().strip() == nom
    nom = vers_candidat(sortie)                 # repli : un nom cité dans le texte
    return (nom, False) if nom else ("INCONNU", False)


tests = {
    '{"locuteur": "ROMEO"}': ("ROMEO", True),
    'Voici : {"locuteur": "Duke"} merci': ("DUKE VINCENTIO", False),
    "Je pense que c'est PETRUCHIO.": ("PETRUCHIO", False),
    "Aucune idée": ("INCONNU", False),
}
for entree, attendu in tests.items():
    assert lire_reponse(entree) == attendu, entree
print(f"lire_reponse : {len(tests)} tests passés")

In [ ]:
def classer_avec_llm(textes, variante, temperature=0.0, max_new_tokens=30):
    """Applique le prompt `variante` à chaque réplique. Renvoie (tableau détaillé, durée en s)."""
    conversations = [construire_prompt(r, variante) for r in textes]
    t0 = time.time()
    sorties = generer(conversations, temperature=temperature, max_new_tokens=max_new_tokens)
    duree = time.time() - t0
    lus = [lire_reponse(s) for s in sorties]
    detail = pd.DataFrame({
        "replique": list(textes),
        "sortie_brute": sorties,
        "pred": [label for label, _ in lus],
        "format_ok": [ok for _, ok in lus],
    })
    return detail, duree


def scores(y_true, y_pred):
    """Métriques macro sur les 5 vraies classes ; une prédiction INCONNU compte comme une erreur."""
    macro = dict(labels=CANDIDATS, average="macro", zero_division=0)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, **macro),
        "recall_macro": recall_score(y_true, y_pred, **macro),
        "f1_macro": f1_score(y_true, y_pred, **macro),
    }

## 5. Comparer les prompts sur la validation

Les trois variantes passent sur les 114 répliques de validation. En plus des métriques, on suit le taux de réponses au bon format, la taille moyenne du prompt en tokens et le temps par requête.

Pour un essai rapide, mettre N_VAL = 30 ; None utilise toute la validation.

In [ ]:
N_VAL = None

X_val_eval = X_val if N_VAL is None else X_val.iloc[:N_VAL]
y_val_eval = y_val.loc[X_val_eval.index]

sorties_val, bilan_val = {}, {}
for variante in VARIANTES:
    detail, duree = classer_avec_llm(X_val_eval.values, variante)
    sorties_val[variante] = detail
    taille_prompt = np.mean([nb_tokens(construire_prompt(r, variante)) for r in X_val_eval.values[:20]])
    bilan_val[variante] = {
        **scores(y_val_eval.values, detail["pred"].values),
        "format_ok": detail["format_ok"].mean(),
        "inconnu": (detail["pred"] == "INCONNU").mean(),
        "tokens_prompt": taille_prompt,
        "s_par_requete": duree / len(detail),
    }
    print(f"{variante:<20} {duree:6.1f} s")

tab_val = pd.DataFrame(bilan_val).T
tab_val.round(3)

In [ ]:
variante_retenue = tab_val["f1_macro"].astype(float).idxmax()
print(f"prompt retenu (meilleur F1 macro en validation) : {variante_retenue}")

ax = tab_val[["accuracy", "f1_macro", "format_ok"]].astype(float).plot.bar(figsize=(8, 3.5), rot=0)
ax.axhline(1 / len(CANDIDATS), color="grey", ls="--", lw=1)
ax.set_title("Les trois prompts sur la validation (pointillés : hasard)")
ax.set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

### 5.1 Lire quelques sorties brutes

Un score ne suffit pas : on lit aussi les réponses. Le format est-il respecté ? Le modèle ajoute-t-il du texte ? Une classe est-elle prédite beaucoup plus souvent qu'elle n'apparaît ?

In [ ]:
detail = sorties_val[variante_retenue]
print("exemples de sorties :\n")
for brut, pred, vrai in list(zip(detail["sortie_brute"], detail["pred"], y_val_eval.values))[:6]:
    marque = "ok " if pred == vrai else "err"
    print(f"  [{marque}] {brut!r:<35} -> {pred:<15} (vrai : {vrai})")

print("\nnombre de prédictions par classe, comparé au nombre réel (validation) :")
pd.DataFrame({
    "prédit": detail["pred"].value_counts(),
    "réel": y_val_eval.value_counts(),
}).fillna(0).astype(int)

## 6. Évaluation finale sur le test (une seule fois)

Seul le prompt retenu passe sur le test. On le compare ensuite à la baseline TF-IDF + régression logistique, entraînée sur le train et évaluée sur le même test. On mesure aussi le temps : un LLM ne se justifie que si le gain de qualité compense son coût.

In [ ]:
N_TEST = None   # None = tout le test

X_test_eval = X_test if N_TEST is None else X_test.iloc[:N_TEST]
y_test_eval = y_test.loc[X_test_eval.index]

detail_test, duree_test = classer_avec_llm(X_test_eval.values, variante_retenue)
scores_llm = scores(y_test_eval.values, detail_test["pred"].values)
print(f"LLM ({variante_retenue}) : {len(detail_test)} requêtes en {duree_test:.1f} s, "
      f"soit {duree_test / len(detail_test):.2f} s par requête ; "
      f"{detail_test['format_ok'].mean():.1%} de réponses au bon format\n")

print(classification_report(y_test_eval.values, detail_test["pred"].values, labels=CANDIDATS, zero_division=0))

etiquettes = CANDIDATS + (["INCONNU"] if (detail_test["pred"] == "INCONNU").any() else [])
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay.from_predictions(
    y_test_eval.values, detail_test["pred"].values, labels=etiquettes, normalize="true",
    xticks_rotation=45, cmap="Blues", colorbar=False, values_format=".2f", ax=ax)
ax.set_title(f"LLM, prompt {variante_retenue}\n(chaque ligne est normalisée : diagonale = rappel)")
plt.tight_layout(); plt.show()

In [ ]:
# baseline classique : TF-IDF + régression logistique, ajustée sur le train
baseline = Pipeline([
    ("vec", TfidfVectorizer(min_df=2, ngram_range=(1, 2), sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=2000, C=10, class_weight="balanced")),
])
baseline.fit(X_train, y_train)

t0 = time.time()
pred_base = baseline.predict(X_test_eval)
duree_base = time.time() - t0
scores_base = scores(y_test_eval.values, pred_base)

# F1 macro (test) du meilleur réseau du TP 2 (TextCNN), recopié à la main depuis ce TP ; None pour l'ignorer
F1_DL_TP2 = 0.57
NOM_DL_TP2 = "TextCNN (TP 2)"

tableau = {
    "TF-IDF + LogReg (entraîné)": {**scores_base, "s_par_requete": duree_base / len(pred_base)},
    f"LLM {variante_retenue} (sans entraînement)": {**scores_llm, "s_par_requete": duree_test / len(detail_test)},
}
if F1_DL_TP2 is not None:
    tableau[NOM_DL_TP2] = {"f1_macro": F1_DL_TP2}

comparaison = pd.DataFrame(tableau).T
comparaison.round(4)

## 7. Analyse des erreurs

Quelles confusions reviennent le plus, et sur quelles répliques ?

In [ ]:
erreurs = detail_test.assign(vrai=y_test_eval.values)
erreurs = erreurs[erreurs["pred"] != erreurs["vrai"]]
print(f"{len(erreurs)} erreurs sur {len(detail_test)} répliques\n")
print("confusions les plus fréquentes (vrai -> prédit) :")
print(erreurs.groupby(["vrai", "pred"]).size().sort_values(ascending=False).head(6).to_string(), "\n")

erreurs.assign(replique=erreurs["replique"].str.slice(0, 90))[["replique", "sortie_brute", "pred", "vrai"]].head(8)

Une réplique courte contient peu d'indices, pour le LLM comme pour un classifieur. On compare donc le taux de bonnes réponses des deux approches par tranche de longueur.

In [ ]:
par_longueur = pd.DataFrame({
    "n_mots": X_test_eval.str.split().str.len().values,
    "LLM": detail_test["pred"].values == y_test_eval.values,
    "TF-IDF": pred_base == y_test_eval.values,
})
par_longueur["tranche"] = pd.cut(par_longueur["n_mots"], bins=[0, 10, 20, 40, 1000],
                                 labels=["5-10", "11-20", "21-40", ">40"])
par_longueur.groupby("tranche", observed=True)[["LLM", "TF-IDF"]].mean().round(3)

## 8. Et si on échantillonne ?

À température 0, la réponse est (presque) déterministe. Dès que la température monte, le même prompt peut donner des réponses différentes, et un système en production doit connaître cette variance. On relance trois fois le prompt retenu à température 0,8 sur les 60 premières répliques de la validation.

In [ ]:
N_VARIAB = min(60, len(X_val_eval))
sous_ens = X_val_eval.iloc[:N_VARIAB]
y_sous = y_val_eval.loc[sous_ens.index].values

f1_t0 = scores(y_sous, sorties_val[variante_retenue]["pred"].values[:N_VARIAB])["f1_macro"]
f1_t08 = []
for essai in range(3):
    d, _ = classer_avec_llm(sous_ens.values, variante_retenue, temperature=0.8)
    f1_t08.append(scores(y_sous, d["pred"].values)["f1_macro"])

print(f"F1 macro à température 0   : {f1_t0:.3f}")
print(f"F1 macro à température 0,8 : {np.round(f1_t08, 3)}  (moyenne {np.mean(f1_t08):.3f}, écart-type {np.std(f1_t08):.3f})")

## 9. Conclusion

Les chiffres ci-dessous viennent de notre exécution sur Colab (GPU T4, Qwen2.5-3B-Instruct). Avec un autre modèle, ou après une nouvelle exécution pour la partie à température 0,8, ils peuvent légèrement changer : les tableaux du §5 au §8 font foi.

1. Sans aucune donnée d'entraînement, une consigne claire suffit à faire mieux que le hasard (20 %). Avec le prompt few_shot, le LLM obtient 0,44 d'accuracy et 0,42 de F1 macro sur le test. Il reste pourtant loin de la baseline entraînée (TF-IDF + régression logistique : 0,64 de F1 macro) et du meilleur réseau du TP 2 (TextCNN : 0,57).

2. Le score global cache un fort biais. Le LLM répond presque toujours « DUKE VINCENTIO » : 74 prédictions sur 114 en validation, pour 25 répliques réellement prononcées par lui. Sur le test, cette classe a un rappel de 0,92 pour une précision de 0,32. À l'inverse, Gloucester n'est presque jamais reconnu (rappel 0,07) et Roméo rarement (rappel 0,19, mais précision 1,00). Il faut donc lire les résultats classe par classe et regarder la matrice de confusion ; l'accuracy seule ne montre rien de tout ça.

3. Ajouter du contexte ne suffit pas, les exemples aident. Avec la description des personnages seule, le F1 macro baisse (0,25 pour zero_shot, 0,20 pour zero_shot_contexte). Les démonstrations le font monter à 0,36 avec few_shot, mais le prompt est alors à peu près 4,7 fois plus long (790 tokens contre 167) et chaque requête prend environ trois fois plus de temps.

4. Une sortie de LLM se vérifie. Le JSON est respecté dans 100 % des cas pour zero_shot et few_shot, et dans 99 % des cas pour zero_shot_contexte (une réponse INCONNU). Le parsing est donc fiable ici, mais on le contrôle quand même à chaque fois. Par ailleurs, dès que la température n'est plus nulle, le résultat change d'un tirage à l'autre : sur trois essais à 0,8, le F1 macro a varié entre 0,28 et 0,40, contre 0,375 à température 0. Pour classer, on reste à température 0.

5. Attention à la contamination. *Tiny Shakespeare* est un texte public, très présent dans les données de pré-entraînement : le LLM a peut-être déjà vu ces répliques. Son score ne mesure donc pas seulement sa capacité à généraliser, et il ne se transposerait pas tel quel à un corpus privé.

6. Le coût compte aussi. Le classifieur TF-IDF répond en une fraction de milliseconde ; le LLM met environ 0,4 s par requête, soit plusieurs ordres de grandeur de plus (colonne s_par_requete). Pour une tâche étroite et un gros volume, le pipeline classique reste le bon choix. Le LLM a plus d'intérêt pour du langage ouvert, de la génération, ou comme brique d'un système hybride.

Pistes pour aller plus loin : un modèle plus gros ou une API, un autre ordre des noms dans la consigne (pour tester un biais vers un nom en particulier), plus d'exemples, ou un fine-tuning léger (LoRA) sur le train.